# 20260723_EDA_2019_전국_시도분리정제
- 작성자: 이정연
- 이슈 #20 참고

In [8]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    needs_llm_rerun,
    numbers_preserved,
)
from src.features.pipeline_common import (  # noqa: E402
    SUBTOTAL_LABEL_PATTERN,
    UNIT_NOTATION_PATTERN,
    assign_labels,
    backfill_major_category_from_medium,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    drop_exact_duplicate_rows,
    get_sido_dir,
    normalize_budget_type,
    realign_major_category_from_bracket_marker,
    select_total_budget_rows,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
)

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

print(f"프로젝트 루트: {PROJECT_ROOT}")

프로젝트 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


## 1. 연도·입력·예외 설정

아래 셀은 복사한 노트북에서 가장 먼저 수정한다. `비예산`, `·` 등을 0으로 처리할지는 해당 연도 원본을 확인한 뒤 결정한다.

In [9]:
YEAR = 2019
PREVIOUS_YEAR = YEAR - 1

CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"
CURRENT_NUM_COL = f"{YEAR}년_예산_num"
PREVIOUS_NUM_COL = f"{PREVIOUS_YEAR}년_예산_num"

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

SOURCE_FILE = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2019년도 지방자치단체 저출산고령사회 시행계획 (제3차 기본계획)_칼럼정렬.xlsx"
)
SOURCE_SHEET = "정리본_자동"
TABLE1_FILE = SOURCE_FILE  # Table 1 시트가 같은 파일 안에 있음

ZERO_BUDGET_TOKENS = ("-",)
# 2019 원본의 충북 구간이 지역 컬럼에서 "충청"으로 잘못 추출되어 있다.
REGION_NAME_NORMALIZATION = {"충청": "충북"}
# 일부 지역의 대분류 제목만 ASCII 로마자로 표기되어 공통 분류 패턴이 인식하지 못한다.
MAJOR_CATEGORY_TITLE_NORMALIZATION = {
    "I. 공통사업": "Ⅰ. 공통사업",
    "II. 자체사업": "Ⅱ. 자체사업",
}
# 2019 중분류는 "1. (공통)제목" 접두형이므로 공통 유틸의 접미형 라벨로 바꾼다.
MEDIUM_CATEGORY_PREFIX_PATTERN = (
    r"^(?P<number>\d+)\.\s*\((?P<budget_type>공통|자체)\)\s*(?P<title>.+?)\s*$"
)
# 2019 강원에만 있는 띄어쓰기 변형을 전국 공통 표기로 통일한다.
MEDIUM_CATEGORY_TITLE_NORMALIZATION = {
    "2. 함께 만들어 가는 행복한 노후(공통사업)": "2. 함께 만들어가는 행복한 노후(공통사업)",
}
# 충남 원본의 자체사업 3번 중분류만 '(공통)'으로 잘못 표기되어 있다.
# 자체사업 합계가 1~3번 소계의 합과 정확히 일치하므로 계층 전파 후 바로잡는다.
CHUNGNAM_MEDIUM_CATEGORY_FIX = {
    "source": "3. 인구변화 적극대비(공통사업)",
    "target": "3. 인구변화 적극대비(자체사업)",
    "expected_rows": 31,
}
# 대구의 공통·자체사업 총괄은 세부사업이 아닌 대분류 요약행이다.
SUMMARY_LABEL_PATTERN = r"^(?:공통사업|자체사업)\s*총괄$"
# 2019는 시도별 "(단위 : 백만원)" 헤더 행과 총계/소계/합계 라벨 행이 있음 (이슈 #24)
EXTRA_HEADER_PATTERNS = (
    UNIT_NOTATION_PATTERN,
    SUBTOTAL_LABEL_PATTERN,
    SUMMARY_LABEL_PATTERN,
)
# 2019는 사업분류재정구분이 계/국비/지방비 등 재원 구분이라 이중계상 위험이 있음 (이슈 #26)
NEEDS_FUNDING_SOURCE_CLEANUP = True
# 경기 3개 사업은 총액 행의 재원구분만 비어 있고 뒤의 국비+지방비 합과 금액이 같다.
BLANK_TOTAL_FINANCE_ROWS = (3618.0, 3621.0, 3624.0)
# 대전 1개 사업의 '계' 앞에 추출 기호가 붙어 공통 재원 정리 유틸이 인식하지 못한다.
FINANCE_TYPE_NORMALIZATION = {"｣계": "계"}
# 원본 표에서 셀 분리 버그로 확인된 완전 중복 행의 원본행 번호만 지정한다(자동 감지 후보는
# classify-and-label 셀에서 별도로 출력·검토한다). 현재 확인된 대상 없음.
CONFIRMED_DUPLICATE_ROWS = ()
QA_TOLERANCE = 0
RUN_LLM = True

CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
QA_PATH = REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv"

## 2. 데이터 로드와 기본 확인

In [10]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"SOURCE_FILE을 실제 파일로 수정하세요: {SOURCE_FILE}")

df_raw = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET)

required_columns = {
    "지역",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
    "원본행",
}
missing_columns = required_columns.difference(df_raw.columns)
if missing_columns:
    raise KeyError(f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

region_name_fix_counts = {
    source: int(df_raw["지역"].eq(source).sum()) for source in REGION_NAME_NORMALIZATION
}
df_raw["지역"] = df_raw["지역"].replace(REGION_NAME_NORMALIZATION)
if df_raw["지역"].isin(REGION_NAME_NORMALIZATION).any():
    raise ValueError("지역명 보정 후에도 원본 오표기가 남아 있습니다.")
if df_raw["지역"].nunique() != 17 or "충북" not in set(df_raw["지역"].dropna()):
    raise ValueError("2019년 지역명 보정 결과가 17개 시도와 일치하지 않습니다.")

print("데이터 크기:", df_raw.shape)
print("지역 수:", df_raw["지역"].nunique())
print("지역명 보정:", region_name_fix_counts, "→", REGION_NAME_NORMALIZATION)
display(df_raw.head())

데이터 크기: (8972, 12)
지역 수: 17
지역명 보정: {'충청': 595} → {'충청': '충북'}


,지역,세부사업명,사업분류재정구분,2019년 예산,2018년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,(단위 : 백만원),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,서울,총계,NaN,5907309,5233824,673485,12.9,NaN,1.0,2.0,4.0,데이터행
2,서울,Ⅰ. 공통사업,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,5.0,구분행사업명 연속 후보
3,서울,공통사업 합계,계,5357484,4716666,640818,13.6,NaN,1.0,2.0,6.0,데이터행
4,서울,공통사업 합계,국비,3465904,2960722,505182,17.1,NaN,1.0,2.0,7.0,데이터행


## 3. 행 분류·계층 전파·숫자 변환

소계 QA에 원본 소계 행의 숫자값도 필요하므로 `df_labeled` 전체에 숫자 컬럼을 만든다.

In [11]:
major_title = df_raw["세부사업명"].astype("string").str.strip()
major_title_fix_counts = {
    source: int(major_title.eq(source).sum()) for source in MAJOR_CATEGORY_TITLE_NORMALIZATION
}
df_raw["세부사업명"] = major_title.replace(MAJOR_CATEGORY_TITLE_NORMALIZATION)
if df_raw["세부사업명"].isin(MAJOR_CATEGORY_TITLE_NORMALIZATION).any():
    raise ValueError("대분류 표준화 후에도 ASCII 로마자 제목이 남아 있습니다.")
print("대분류 제목 표준화:", major_title_fix_counts)

if NEEDS_FUNDING_SOURCE_CLEANUP:
    finance_type = df_raw["사업분류재정구분"].astype("string").str.strip()
    finance_type_fix_counts = {
        source: int(finance_type.eq(source).sum()) for source in FINANCE_TYPE_NORMALIZATION
    }
    if finance_type_fix_counts != {"｣계": 1}:
        raise ValueError("2019년 대전 비표준 계 표기 보정 대상을 다시 확인하세요.")
    df_raw["사업분류재정구분"] = finance_type.replace(FINANCE_TYPE_NORMALIZATION)

    # 확정된 경기 빈칸 총액 행만 보정한다. 각 행의 당해·전년도 금액이 바로 뒤
    # 국비·지방비 합과 모두 일치하지 않으면 원본 구조가 달라진 것이므로 중단한다.
    for total_row_id in BLANK_TOTAL_FINANCE_ROWS:
        total_positions = df_raw.index[df_raw["원본행"].eq(total_row_id)].tolist()
        if len(total_positions) != 1:
            raise ValueError(f"경기 빈칸 총액 원본행을 다시 확인하세요: {total_row_id}")
        total_position = total_positions[0]
        total_row = df_raw.loc[total_position]
        funding_rows = df_raw.loc[total_position + 1 : total_position + 2]
        funding_tokens = funding_rows["사업분류재정구분"].astype("string").str.strip()
        same_project = (
            funding_rows["지역"].eq(total_row["지역"])
            & funding_rows["머리글행"].eq(total_row["머리글행"])
            & funding_rows["세부사업명"].eq(total_row["세부사업명"])
        ).all()
        if (
            pd.notna(total_row["사업분류재정구분"])
            or set(funding_tokens.dropna()) != {"국비", "지방비"}
            or not same_project
        ):
            raise ValueError(f"경기 빈칸 총액 재원 구조를 다시 확인하세요: {total_row_id}")
        for budget_column in [CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL]:
            total_budget = to_numeric_budget(
                pd.Series([total_row[budget_column]]), zero_tokens=ZERO_BUDGET_TOKENS
            ).iloc[0]
            funding_sum = to_numeric_budget(
                funding_rows[budget_column], zero_tokens=ZERO_BUDGET_TOKENS
            ).sum(min_count=1)
            if total_budget != funding_sum:
                raise ValueError(
                    f"경기 빈칸 총액과 국비·지방비 합이 다릅니다: "
                    f"원본행={total_row_id}, 컬럼={budget_column}"
                )
        df_raw.loc[total_position, "사업분류재정구분"] = "계"
    print(
        "재원구분 계 표준화:",
        {"경기_빈칸총액": len(BLANK_TOTAL_FINANCE_ROWS), **finance_type_fix_counts},
    )

    # 셀 분리 추출로 바로 붙은 완전 동일 행만 중복으로 확정한다.
    # 사업명과 예산만 같은 정상 사업은 제거하지 않는다.
    duplicate_compare_cols = [column for column in df_raw.columns if column != "원본행"]
    row_fingerprint = pd.util.hash_pandas_object(df_raw[duplicate_compare_cols], index=False)
    is_adjacent_exact_duplicate = row_fingerprint.eq(row_fingerprint.shift())
    adjacent_duplicate_rows = df_raw.loc[is_adjacent_exact_duplicate, "원본행"].tolist()
    confirmed_duplicate_rows = set(CONFIRMED_DUPLICATE_ROWS) | set(adjacent_duplicate_rows)
    print("연속 완전 중복 제거 후보:", len(adjacent_duplicate_rows))
    display(
        df_raw.loc[is_adjacent_exact_duplicate]
        .groupby("지역")
        .size()
        .rename("제거_후보_행수")
        .to_frame()
    )
    df_raw = drop_exact_duplicate_rows(df_raw, confirmed_duplicate_rows=confirmed_duplicate_rows)
    df_raw = select_total_budget_rows(
        df_raw,
        budget_cols=[CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL],
        zero_tokens=ZERO_BUDGET_TOKENS,
    )
    print("재원구분(계/국비/지방비) 정리 후 행 수:", len(df_raw))

# 2020 노트북처럼 연도 고유 중분류 제목을 먼저 식별하되, 2019 접두형을
# 공통 classify_row/backfill_major_category_from_medium가 재사용할 수 있는 접미형으로 표준화한다.
detail_name = df_raw["세부사업명"].astype("string").str.strip()
medium_title_parts = detail_name.str.extract(MEDIUM_CATEGORY_PREFIX_PATTERN)
medium_category_mask = medium_title_parts.notna().all(axis=1)
normalized_medium_title = (
    medium_title_parts["number"]
    + ". "
    + medium_title_parts["title"].str.strip()
    + "("
    + medium_title_parts["budget_type"]
    + "사업)"
)
df_raw.loc[medium_category_mask, "세부사업명"] = normalized_medium_title.loc[medium_category_mask]
medium_title_fix_counts = {
    source: int(df_raw["세부사업명"].eq(source).sum())
    for source in MEDIUM_CATEGORY_TITLE_NORMALIZATION
}
df_raw["세부사업명"] = df_raw["세부사업명"].replace(MEDIUM_CATEGORY_TITLE_NORMALIZATION)
if medium_title_fix_counts != {"2. 함께 만들어 가는 행복한 노후(공통사업)": 1}:
    raise ValueError("2019년 강원 중분류 띄어쓰기 보정 대상을 다시 확인하세요.")
if int(medium_category_mask.sum()) != 98:
    raise ValueError("2019년 중분류 제목은 98건이어야 합니다.")
if (
    df_raw["세부사업명"]
    .astype("string")
    .str.contains(r"^\d+\.\s*\((?:공통|자체)\)", regex=True, na=False)
    .any()
):
    raise ValueError("중분류 표준화 후에도 접두형 라벨이 남아 있습니다.")
print(
    "중분류 제목 표준화:",
    int(medium_category_mask.sum()),
    "건 / 띄어쓰기 보정:",
    medium_title_fix_counts,
)

df_raw["사업행구분"] = df_raw["세부사업명"].apply(
    lambda value: classify_row(value, extra_header_patterns=EXTRA_HEADER_PATTERNS)
)

# TODO: 2019만의 계층 예외(제목 접미사 누락, 대괄호/괄호닫힘 세부그룹 등)가 있으면
# assign_labels 전에 여기서 df_raw["사업행구분"]·df_raw["세부사업명"]을 보정한다.
# 2016·2017 노트북의 classify-and-label 셀을 참고한다.

df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# 대분류 로마숫자 헤더가 없는 지역(예: 광주)의 대분류를 중분류 라벨에서 역으로 채운다.
df_labeled = backfill_major_category_from_medium(df_labeled)
# 로마숫자가 대분류가 아니라 중분류(주제) 축인 지역(예: 대전)의 대분류를 대괄호 표기에서 재정렬한다.
df_labeled = realign_major_category_from_bracket_marker(df_labeled)

# 충남 원본에는 자체사업 3번 중분류가 '(공통)'으로 잘못 표기되어 있다. 2019년 자체사업
# 소계 269,484 + 54,309 + 877 = 자체사업 합계 324,670이고, 2018년 금액도
# 90,654 + 41,152 + 871 = 132,677로 일치하므로 정제 계층만 자체사업으로 보정한다.
chungnam_medium_fix_mask = (
    df_labeled["지역"].eq("충남")
    & df_labeled["대분류"].eq("Ⅱ. 자체사업")
    & df_labeled["중분류"].eq(CHUNGNAM_MEDIUM_CATEGORY_FIX["source"])
)
if int(chungnam_medium_fix_mask.sum()) != CHUNGNAM_MEDIUM_CATEGORY_FIX["expected_rows"]:
    raise ValueError("2019년 충남 중분류 원본 오표기 보정 대상을 다시 확인하세요.")
df_labeled.loc[chungnam_medium_fix_mask, "중분류"] = CHUNGNAM_MEDIUM_CATEGORY_FIX["target"]
print("충남 중분류 원본 오표기 보정:", int(chungnam_medium_fix_mask.sum()), "행")

# TODO: 위 유틸과 연도별 확정 보정으로도 안 잡히는 지역별 계층 예외가 있으면 여기서 보정한다.

df_labeled[CURRENT_NUM_COL] = to_numeric_budget(
    df_labeled[CURRENT_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)
df_labeled[PREVIOUS_NUM_COL] = to_numeric_budget(
    df_labeled[PREVIOUS_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)

df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
if df_leaf[["대분류", "중분류"]].isna().any().any():
    missing_hierarchy = df_leaf.loc[
        df_leaf[["대분류", "중분류"]].isna().any(axis=1),
        ["지역", "원본행", "세부사업명", "대분류", "중분류"],
    ]
    raise ValueError(f"대·중분류 결측이 남아 있습니다: {len(missing_hierarchy)}건")
if df_leaf["세부사업명"].isin(df_raw.loc[medium_category_mask, "세부사업명"]).any():
    raise ValueError("중분류 제목이 세부사업에 남아 있습니다.")
print(df_labeled["사업행구분"].value_counts())
print("세부사업 행 수:", len(df_leaf))

대분류 제목 표준화: {'I. 공통사업': 3, 'II. 자체사업': 3}
재원구분 계 표준화: {'경기_빈칸총액': 3, '｣계': 1}
연속 완전 중복 제거 후보: 0


,제거_후보_행수
지역,


재원구분(계/국비/지방비) 정리 후 행 수: 6774
중분류 제목 표준화: 98 건 / 띄어쓰기 보정: {'2. 함께 만들어 가는 행복한 노후(공통사업)': 1}
충남 중분류 원본 오표기 보정: 31 행
사업행구분
세부사업      6484
헤더반복       158
중분류_소계      98
대분류_소계      34
Name: count, dtype: int64
세부사업 행 수: 6484


In [12]:
# 지역마다 유니크 대분류·중분류가 몇 개고 종류가 무엇인지 확인
category_by_region = df_leaf.groupby("지역").agg(
    대분류_개수=("대분류", "nunique"),
    대분류_종류=("대분류", lambda s: sorted(s.dropna().unique())),
    중분류_개수=("중분류", "nunique"),
    중분류_종류=("중분류", lambda s: sorted(s.dropna().unique())),
)

category_by_region

,대분류_개수,대분류_종류,중분류_개수,중분류_종류
지역,,,,
강원,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],6,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(공통사업)' '3. 인구변화 적극대비(자체사업)']
경기,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],6,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(공통사업)' '3. 인구변화 적극대비(자체사업)']
경남,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],6,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(공통사업)' '3. 인구변화 적극대비(자체사업)']
경북,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],6,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(공통사업)' '3. 인구변화 적극대비(자체사업)']
광주,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],5,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(자체사업)']
대구,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],5,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(자체사업)']
대전,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],6,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(공통사업)' '3. 인구변화 적극대비(자체사업)']
부산,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],6,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(공통사업)' '3. 인구변화 적극대비(자체사업)']
서울,2,['Ⅰ. 공통사업' 'Ⅱ. 자체사업'],6,['1. 함께 돌보고 함께 일하는 사회(공통사업)' '1. 함께 돌보고 함께 일하는 사회(자체사업)' '2. 함께 만들어가는 행복한 노후(공통사업)' '2. 함께 만들어가는 행복한 노후(자체사업)' '3. 인구변화 적극대비(공통사업)' '3. 인구변화 적극대비(자체사업)']


## 4. 텍스트 정제와 예산 재계산

In [13]:
for column in ["세부사업명", "대분류", "중분류"]:
    df_leaf[column] = clean_text(df_leaf[column])

df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

# 이전 연도와 동일하게 원본의 계/국비/지방비는 재원구분에 보존하고,
# 최종 사업분류재정구분은 대분류의 공통/자체 축에서 다시 생성한다.
df_leaf["재원구분"] = clean_text(df_leaf["재원구분"])
df_leaf["사업분류재정구분"] = pd.Series(pd.NA, index=df_leaf.index, dtype="string")
is_common_leaf = df_leaf["대분류"].str.contains("공통", na=False)
is_self_leaf = df_leaf["대분류"].str.contains("자체", na=False)
df_leaf.loc[is_common_leaf, "사업분류재정구분"] = "공통"
df_leaf.loc[is_self_leaf, "사업분류재정구분"] = "자체"

budget_type_missing = df_leaf["사업분류재정구분"].isna()
budget_type_mismatch = (is_common_leaf & ~df_leaf["사업분류재정구분"].eq("공통")) | (
    is_self_leaf & ~df_leaf["사업분류재정구분"].eq("자체")
)
if budget_type_missing.any() or budget_type_mismatch.any():
    raise ValueError(
        "대분류 기반 사업분류재정구분 생성에 실패했습니다: "
        f"결측={int(budget_type_missing.sum())}, "
        f"불일치={int(budget_type_mismatch.sum())}"
    )
print("사업분류재정구분:", df_leaf["사업분류재정구분"].value_counts().to_dict())

budget_changes = calculate_budget_changes(
    df_leaf[CURRENT_NUM_COL],
    df_leaf[PREVIOUS_NUM_COL],
)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
display(df_leaf[["지역", "세부사업명", "당해예산", "전년도예산", "증감액", "증감율"]].head())

사업분류재정구분: {'자체': 5441, '공통': 1043}


,지역,세부사업명,당해예산,전년도예산,증감액,증감율
6276,강원,횡성 꿈키움 통합복지센터 (꿈키움취업지원센터 조성),980,0,980,<NA>
6277,강원,강원도형 청년일자리 지원,25963,8284,17679,213.4
6278,강원,다함께 돌봄센터 운영지원,420,38,382,1005.3
6279,강원,다함께 돌봄센터 기능보강,556,20,536,2680.0
6280,강원,육아종합지원센터 운영 지원,442,456,-14,-3.1


## 5. 중분류 소계 QA와 저장

QA 계산은 공통 함수가 담당하고, 연도별 파일 저장은 노트북에서 담당한다.

In [14]:
qa = build_subtotal_qa(
    df_labeled,
    budget_col=CURRENT_NUM_COL,
    tolerance=QA_TOLERANCE,
    subtotal_label_col="세부사업명",
    subtotal_labels=("소계",),
)

display(qa["결과"].value_counts(dropna=False))
display(qa["원본_소계출처"].value_counts(dropna=False))
display(qa[qa["결과"] == "불일치"])
display(qa[qa["결과"] == "판정불가"])

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
qa.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print(f"QA 저장 완료: {QA_PATH}")

결과
일치      60
불일치     30
판정불가     8
Name: count, dtype: int64

원본_소계출처
별도소계행     90
중분류제목행     8
Name: count, dtype: int64

,지역,대분류,중분류,원본_소계값,원본_소계출처,leaf_합계,차이,절대차이,오차율(%),절대오차율(%),QA_병합상태,결과,허용기준결과
0,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),640662,별도소계행,640538,-124,124,-0.02,0.02,양쪽존재,불일치,허용
3,강원,Ⅱ. 자체사업,1. 함께 돌보고 함께 일하는 사회(자체사업),116209,별도소계행,116210,1,1,0.0,0.0,양쪽존재,불일치,허용
9,경기,Ⅱ. 자체사업,1. 함께 돌보고 함께 일하는 사회(자체사업),806708,별도소계행,806715,7,7,0.0,0.0,양쪽존재,불일치,허용
10,경기,Ⅱ. 자체사업,2. 함께 만들어가는 행복한 노후(자체사업),520135,별도소계행,520136,1,1,0.0,0.0,양쪽존재,불일치,허용
12,경남,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),1281969,별도소계행,1282321,352,352,0.03,0.03,양쪽존재,불일치,허용
15,경남,Ⅱ. 자체사업,1. 함께 돌보고 함께 일하는 사회(자체사업),231780,별도소계행,231781,1,1,0.0,0.0,양쪽존재,불일치,허용
16,경남,Ⅱ. 자체사업,2. 함께 만들어가는 행복한 노후(자체사업),40357,별도소계행,40358,1,1,0.0,0.0,양쪽존재,불일치,허용
19,경북,Ⅰ. 공통사업,2. 함께 만들어가는 행복한 노후(공통사업),1249773,별도소계행,1249772,-1,1,-0.0,0.0,양쪽존재,불일치,허용
21,경북,Ⅱ. 자체사업,1. 함께 돌보고 함께 일하는 사회(자체사업),117356,별도소계행,117359,3,3,0.0,0.0,양쪽존재,불일치,허용
26,광주,Ⅱ. 자체사업,1. 함께 돌보고 함께 일하는 사회(자체사업),52669,별도소계행,52668,-1,1,-0.0,0.0,양쪽존재,불일치,허용


,지역,대분류,중분류,원본_소계값,원본_소계출처,leaf_합계,차이,절대차이,오차율(%),절대오차율(%),QA_병합상태,결과,허용기준결과
8,경기,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,39,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가
14,경남,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,39,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가
20,경북,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,39,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가
48,서울,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,44,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가
60,울산,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,39,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가
66,인천,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,39,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가
72,전남,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,39,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가
94,충북,Ⅰ. 공통사업,3. 인구변화 적극대비(공통사업),<NA>,중분류제목행,39,<NA>,<NA>,<NA>,<NA>,양쪽존재,판정불가,판정불가


QA 저장 완료: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/reports/2019_전국_QA_검증결과.csv


## 6. 주요내용 라벨 정리·부분 추출·유사 사업명 후보

In [15]:
df_leaf["주요내용"] = df_leaf["주요내용"].apply(dedup_label)
df_leaf["주요내용_패턴"] = df_leaf["주요내용"].apply(check_pattern)
df_leaf["주요내용_패턴_확장"] = df_leaf["주요내용"].apply(check_pattern_broad)

regex_extracted = pd.DataFrame(
    df_leaf["주요내용"].apply(extract_via_regex).tolist(),
    index=df_leaf.index,
)
df_leaf["지원대상"] = regex_extracted["지원대상"]
df_leaf["지원내용_상세"] = regex_extracted["지원내용"]

near_duplicates = find_near_duplicates(df_leaf, threshold=90)
print("유사 사업명 검토 후보:", len(near_duplicates))
display(near_duplicates.head(20))

유사 사업명 검토 후보: 223


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
0,경남,1. 함께 돌보고 함께 일하는 사회(자체사업),산모신생아 건강관리 본인부담금 지원,120,본인부담금 90%를 현금으로 지원(서비스 종 료 후 환급),산모･신생아 건강관리 본인부담금 지원,120,본인부담금의 90% 지원,97.435897
1,경남,1. 함께 돌보고 함께 일하는 사회(자체사업),산모･신생아 건강관리 본인부담금 지원,120,본인부담금의 90% 지원,산모신생아 건강관리 본인부담금 지원,100,"첫째아 50%, 둘째아 80%, 셋째아 90% - 쌍태아 이상 소득구간 80 이하 90%, 80% 초과 80% 최대 100만원까지 지원",97.435897
2,경남,2. 함께 만들어가는 행복한 노후(자체사업),4대 이상 가정 효도수당 지원,20,4대 이상 가정에 설/명절 효도수당 각 50만원 씩 지급,4세대 이상 가정 효도수당 지원,16,거창군에 주소를 둔 4대 이상 거주가정 매월 10만원 지원,96.969697
3,전남,2. 함께 만들어가는 행복한 노후(자체사업),노인 목욕비 및 이･미용비 지원,2346,어르신 이용권 지원,노인 목욕 및 이･미용비 지원,740,노인 목욕 및 이‧미용비 지원,96.969697
4,강원,2. 함께 만들어가는 행복한 노후(자체사업),저소득 노인 건강보험료 지원,77,저소득 노인가구 건강보험료 지원,저소득층 노인 건강보험료 지원,108,건강보험료 부과금액이 '최저보험료 이하'인 65세 이상 노인세대 건강보험료 지원,96.774194
5,강원,2. 함께 만들어가는 행복한 노후(자체사업),저소득층 노인 건강보험료 지원,108,건강보험료 부과금액이 '최저보험료 이하'인 65세 이상 노인세대 건강보험료 지원,저소득 노인 건강보험료 지원,66,"국민건강보험료 월 15,000원 이하인 만 65세 노인가구 보험료 지원",96.774194
6,경기,3. 인구변화 적극대비(자체사업),생애주기 맞춤형 인구교육,35,"아동, 청소년, 대학생 대상 인구위기 인식개선 을 위한 인구교육 실시",생애주기별 맞춤형 인구교육,10,생애주기별 맞춤형 인구교육,96.296296
7,경기,1. 함께 돌보고 함께 일하는 사회(자체사업),다자녀가정 수도요금 감면,0,"만 18세 미만의 자녀가 3명 이상 있는 가정에 월 10톤에 해당하는 수도 사용료 감면(2019 조례 개정, 2020년 시행 예정)",다자녀가정 하수도요금 감면,0,만 18세 미만의 자녀가 3명 이상 있는 가정에 월 10톤에 해당하는 하수도사용료 감면(2019 하반기 예정),96.296296
8,부산,1. 함께 돌보고 함께 일하는 사회(자체사업),어린이집 냉난방비 지원,84,어린이집 냉난방비 지원,어린이집 냉･난방비 지원,182,관내 어린이집 냉･난방비 지원,96.000000
9,부산,1. 함께 돌보고 함께 일하는 사회(자체사업),어린이집 냉･난방비 지원,22,어린이집 40개소 냉･난방비 지원,어린이집 냉난방비 지원,84,어린이집 냉난방비 지원,96.000000


## 7. 원본 Table 1 대조(필요 시)

원본 파일의 컬럼 배치가 다르면 `column_indices`를 해당 연도에 맞게 변경한다.

In [16]:
# 예시: 실제 행 번호로 교체한 뒤 주석을 해제한다.
# df_table1 = pd.read_excel(TABLE1_FILE, sheet_name="Table 1", header=None)
# display(show_table1_around(df_table1, center_excel_row=100, window=3, label="원본 대조"))

## 8. LLM 보존형 교정(선택)

`RUN_LLM=True`로 바꾸면 전체 세부사업을 순차 처리한다. 100건마다 체크포인트를 저장하므로 실행이 중단돼도 완료된 결과를 재사용한다.

In [17]:
if RUN_LLM:
    import os
    from functools import partial

    import yaml
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("UPSTAGE_API_KEY")
    if not api_key:
        raise RuntimeError("UPSTAGE_API_KEY가 없습니다.")

    with (CONFIG_DIR / "llm_extraction.yaml").open(encoding="utf-8") as file:
        llm_cfg = yaml.safe_load(file)

    client = OpenAI(api_key=api_key, base_url=llm_cfg["upstage"]["base_url"])
    call_once = partial(call_llm_once, client=client, llm_config=llm_cfg)

    print("LLM 연결 설정 완료:", llm_cfg["upstage"]["model"])
else:
    print("RUN_LLM=False: LLM 호출을 건너뜁니다.")

LLM 연결 설정 완료: solar-pro3


In [18]:
# 2019 전체 leaf LLM 교정: 순차 실행, 100건 단위 체크포인트, 숫자 보존 QA
if RUN_LLM:
    from concurrent.futures import ThreadPoolExecutor

    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    if CHECKPOINT_PATH.exists():
        checkpoint_df = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig", index_col=0)
        checkpoint_df.index = pd.to_numeric(checkpoint_df.index, errors="raise").astype(int)
        checkpoint_df = checkpoint_df.loc[checkpoint_df.index.intersection(df_leaf.index)]
        rerun_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
        checkpoint_df = checkpoint_df.loc[~rerun_mask].copy()
        print(f"체크포인트: {len(checkpoint_df)}건 유지, {int(rerun_mask.sum())}건 재실행")
    else:
        checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])

    completed_index = set(checkpoint_df.index)
    targets = [idx for idx in df_leaf.index if idx not in completed_index]
    print(f"LLM 처리 대상: {len(targets)}건 / 전체 {len(df_leaf)}건")

    def clean_target(idx):
        row = df_leaf.loc[idx]
        if pd.isna(row["주요내용"]):
            return idx, None
        # API 실패를 완료로 저장하지 않고 중단해 다음 실행에서 해당 행부터 재개한다.
        cleaned = call_once(row["세부사업명"], str(row["주요내용"]))
        if cleaned is None:
            raise RuntimeError(f"LLM 응답 파싱 실패: index={idx}, 원본행={row['원본행']}")
        return idx, cleaned

    pending = {}
    with ThreadPoolExecutor(max_workers=1) as executor:
        for processed, (idx, cleaned) in enumerate(executor.map(clean_target, targets), 1):
            pending[idx] = cleaned
            if processed % 100 == 0:
                partial_results = pd.DataFrame.from_dict(
                    pending, orient="index", columns=["주요내용_정제"]
                )
                checkpoint_df = pd.concat([checkpoint_df, partial_results])
                checkpoint_df = checkpoint_df.loc[~checkpoint_df.index.duplicated(keep="last")]
                checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
                pending = {}
                print(f"체크포인트 저장: {processed}/{len(targets)}건")

    if pending:
        partial_results = pd.DataFrame.from_dict(pending, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial_results])

    checkpoint_df = checkpoint_df.loc[~checkpoint_df.index.duplicated(keep="last")]
    checkpoint_df = checkpoint_df.reindex(df_leaf.index)
    df_leaf["주요내용_정제"] = checkpoint_df["주요내용_정제"]
    # LLM이 다시 만든 독립 불릿은 제거하되 단어 내부 가운데점은 보존한다.
    df_leaf["주요내용_정제"] = df_leaf["주요내용_정제"].apply(dedup_label)
    checkpoint_df["주요내용_정제"] = df_leaf["주요내용_정제"]
    checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")

    df_leaf["숫자보존"] = [
        numbers_preserved(original, cleaned)
        for original, cleaned in zip(df_leaf["주요내용"], df_leaf["주요내용_정제"])
    ]
    bad_index = df_leaf.index[~df_leaf["숫자보존"]]
    if len(bad_index):
        review_cols = ["지역", "원본행", "세부사업명", "주요내용", "주요내용_정제"]
        df_leaf.loc[bad_index, review_cols].to_csv(
            REPORTS_DIR / f"{YEAR}_LLM_숫자불일치_검토.csv",
            index=False,
            encoding="utf-8-sig",
        )
        df_leaf.loc[bad_index, "주요내용_정제"] = df_leaf.loc[bad_index, "주요내용"]
        checkpoint_df.loc[bad_index, "주요내용_정제"] = df_leaf.loc[bad_index, "주요내용"]

    missing_cleaned = df_leaf["주요내용"].notna() & df_leaf["주요내용_정제"].isna()
    df_leaf.loc[missing_cleaned, "주요내용_정제"] = df_leaf.loc[missing_cleaned, "주요내용"]
    checkpoint_df["주요내용_정제"] = df_leaf["주요내용_정제"]
    checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
    print(f"숫자 불일치 원문 복구: {len(bad_index)}건")
    print(f"정제문 결측 원문 복구: {int(missing_cleaned.sum())}건")
else:
    df_leaf["주요내용_정제"] = df_leaf["주요내용"]

LLM 처리 대상: 6484건 / 전체 6484건
체크포인트 저장: 100/6484건
체크포인트 저장: 200/6484건
체크포인트 저장: 300/6484건
체크포인트 저장: 400/6484건
체크포인트 저장: 500/6484건
체크포인트 저장: 600/6484건
체크포인트 저장: 700/6484건
체크포인트 저장: 800/6484건
체크포인트 저장: 900/6484건
체크포인트 저장: 1000/6484건
체크포인트 저장: 1100/6484건
체크포인트 저장: 1200/6484건
체크포인트 저장: 1300/6484건
체크포인트 저장: 1400/6484건
체크포인트 저장: 1500/6484건
체크포인트 저장: 1600/6484건
체크포인트 저장: 1700/6484건
체크포인트 저장: 1800/6484건
체크포인트 저장: 1900/6484건
체크포인트 저장: 2000/6484건
체크포인트 저장: 2100/6484건
체크포인트 저장: 2200/6484건
체크포인트 저장: 2300/6484건
체크포인트 저장: 2400/6484건
체크포인트 저장: 2500/6484건
체크포인트 저장: 2600/6484건
체크포인트 저장: 2700/6484건
체크포인트 저장: 2800/6484건
체크포인트 저장: 2900/6484건
체크포인트 저장: 3000/6484건
체크포인트 저장: 3100/6484건
체크포인트 저장: 3200/6484건
체크포인트 저장: 3300/6484건
체크포인트 저장: 3400/6484건
체크포인트 저장: 3500/6484건
체크포인트 저장: 3600/6484건
체크포인트 저장: 3700/6484건
체크포인트 저장: 3800/6484건
체크포인트 저장: 3900/6484건
체크포인트 저장: 4000/6484건
체크포인트 저장: 4100/6484건
체크포인트 저장: 4200/6484건
체크포인트 저장: 4300/6484건
체크포인트 저장: 4400/6484건
체크포인트 저장: 4500/6484건
체크포인트 저장: 4600/6484건
체크포인트 저장: 4700/

In [19]:
# 숫자 보존 검사가 놓칠 수 있는 프롬프트 라벨 추가·과도한 축약을 복구한다.
original_text = df_leaf["주요내용"].fillna("").astype(str)
cleaned_text = df_leaf["주요내용_정제"].fillna("").astype(str)
length_ratio = cleaned_text.str.len() / original_text.str.len().replace(0, pd.NA)
added_prompt_label = cleaned_text.str.contains("원문:", regex=False) & ~original_text.str.contains(
    "원문:", regex=False
)
semantic_bad = original_text.ne("") & (
    added_prompt_label | length_ratio.lt(0.5) | length_ratio.gt(1.5)
)
if semantic_bad.any():
    semantic_review = df_leaf.loc[
        semantic_bad,
        ["지역", "원본행", "세부사업명", "주요내용", "주요내용_정제"],
    ].copy()
    semantic_review["길이비"] = length_ratio.loc[semantic_bad]
    semantic_review.to_csv(
        REPORTS_DIR / f"{YEAR}_LLM_의미보존_검토.csv",
        index=False,
        encoding="utf-8-sig",
    )
    df_leaf.loc[semantic_bad, "주요내용_정제"] = df_leaf.loc[semantic_bad, "주요내용"]
    if RUN_LLM:
        checkpoint_df.loc[semantic_bad, "주요내용_정제"] = df_leaf.loc[semantic_bad, "주요내용"]
        checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
print(f"의미 보존 이상 원문 복구: {int(semantic_bad.sum())}건")

의미 보존 이상 원문 복구: 2건


## 9. 시도별 최종 저장

최종 컬럼은 해당 연도 작업에서 확정한 스키마로 수정한다. 같은 파일을 중간과 최종 단계에서 중복 저장하지 않는다.

In [20]:
df_leaf["연도"] = YEAR

output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]
missing_output_cols = set(output_cols).difference(df_leaf.columns)
if missing_output_cols:
    raise KeyError(f"최종 저장 전에 필요한 컬럼을 확인하세요: {sorted(missing_output_cols)}")

for sido, group in df_leaf.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv"
    raw_output_path = sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv"
    group[output_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    df_labeled.loc[df_labeled["지역"].eq(sido)].to_csv(
        raw_output_path, index=False, encoding="utf-8-sig"
    )
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

강원: 428행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/강원/2019_강원_세부사업_정제.csv
경기: 1023행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경기/2019_경기_세부사업_정제.csv
경남: 583행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경남/2019_경남_세부사업_정제.csv
경북: 629행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경북/2019_경북_세부사업_정제.csv
광주: 293행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/광주/2019_광주_세부사업_정제.csv
대구: 235행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대구/2019_대구_세부사업_정제.csv
대전: 192행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대전/2019_대전_세부사업_정제.csv
부산: 556행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/부산/2019_부산_세부사업_정제.csv
서울: 162행 저장 -> /Users/leejungyeon/Workspace/projects/한국재저

## 10. Long 포맷 변환 및 저장

당해예산과 전년도예산을 연도별 행으로 변환하고 시도별 파일로 저장한다.

In [21]:
long_id_cols = [column for column in output_cols if column not in {"당해예산", "전년도예산"}]
df_long = df_leaf[output_cols].melt(
    id_vars=long_id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

previous_mask = df_long["예산구분"].eq("전년도예산")
df_long.loc[previous_mask, "연도"] -= 1
df_long.loc[previous_mask, ["증감액", "증감율"]] = None
df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

assert len(df_long) == len(df_leaf) * 2
print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf), "행 x 2)")
display(df_long.head(10))

long 변환 결과: 12968 행 (wide 6484 행 x 2)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,주요내용_정제,증감액,증감율,원본행,지원대상,지원내용_상세,예산구분,예산액
0,2019,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,횡성 꿈키움 통합복지센터 (꿈키움취업지원센터 조성),"생애주기별 원스톱 지역거점센터구축, 새일 센터(횡성군 여성회관)","생애주기별 원스톱 지역거점센터구축, 새일 센터(횡성군 여성회관)",980,<NA>,4738.0,NaN,NaN,당해예산,980
1,2018,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,횡성 꿈키움 통합복지센터 (꿈키움취업지원센터 조성),"생애주기별 원스톱 지역거점센터구축, 새일 센터(횡성군 여성회관)","생애주기별 원스톱 지역거점센터구축, 새일 센터(횡성군 여성회관)",<NA>,<NA>,4738.0,NaN,NaN,전년도예산,0
2,2019,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,강원도형 청년일자리 지원,"도내 만 39세 이하 청년 정규직 인건비, 창업 지원비, 일경험 인건비 지원","도내 만 39세 이하 청년 정규직 인건비, 창업 지원비, 일경험 인건비 지원",17679,213.4,4741.0,NaN,NaN,당해예산,25963
3,2018,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,강원도형 청년일자리 지원,"도내 만 39세 이하 청년 정규직 인건비, 창업 지원비, 일경험 인건비 지원","도내 만 39세 이하 청년 정규직 인건비, 창업 지원비, 일경험 인건비 지원",<NA>,<NA>,4741.0,NaN,NaN,전년도예산,8284
4,2019,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,다함께 돌봄센터 운영지원,다함께 돌봄센터 운영비 지원(10개소),다함께 돌봄센터 운영비 지원(10개소),382,1005.3,4744.0,NaN,NaN,당해예산,420
5,2018,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,다함께 돌봄센터 운영지원,다함께 돌봄센터 운영비 지원(10개소),다함께 돌봄센터 운영비 지원(10개소),<NA>,<NA>,4744.0,NaN,NaN,전년도예산,38
6,2019,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,다함께 돌봄센터 기능보강,다함께 돌봄센터 설치비 지원(9개소),다함께 돌봄센터 설치비 지원(9개소),536,2680.0,4747.0,NaN,NaN,당해예산,556
7,2018,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,다함께 돌봄센터 기능보강,다함께 돌봄센터 설치비 지원(9개소),다함께 돌봄센터 설치비 지원(9개소),<NA>,<NA>,4747.0,NaN,NaN,전년도예산,20
8,2019,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,육아종합지원센터 운영 지원,센터 인건비 및 운영비(도 및 시･군 1개소),센터 인건비 및 운영비(도 및 시･군 1개소),-14,-3.1,4750.0,NaN,NaN,당해예산,442
9,2018,강원,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),공통,육아종합지원센터 운영 지원,센터 인건비 및 운영비(도 및 시･군 1개소),센터 인건비 및 운영비(도 및 시･군 1개소),<NA>,<NA>,4750.0,NaN,NaN,전년도예산,456


In [22]:
for sido, group in df_long.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제_long.csv"
    group.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

강원: 856행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/강원/2019_강원_세부사업_정제_long.csv
경기: 2046행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경기/2019_경기_세부사업_정제_long.csv
경남: 1166행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경남/2019_경남_세부사업_정제_long.csv
경북: 1258행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경북/2019_경북_세부사업_정제_long.csv
광주: 586행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/광주/2019_광주_세부사업_정제_long.csv
대구: 470행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대구/2019_대구_세부사업_정제_long.csv
대전: 384행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대전/2019_대전_세부사업_정제_long.csv
부산: 1112행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/부산/2019_부산_세부사업_정제_long.csv
서울: 324행 저장 -> /User

## 완료 전 체크리스트

- [ ] 17개 시도 포함 여부 확인
- [ ] 행 유형 분포 확인
- [ ] 연도 고유 헤더·연속행 후보 원본 대조
- [ ] 재원구분(계/국비/지방비) 이중계상 여부 확인 및 `NEEDS_FUNDING_SOURCE_CLEANUP` 반영 (2016~2019)
- [ ] 중분류 소계 QA 불일치 원인 기록
- [ ] 증감액·증감율 재계산 결과 확인
- [ ] LLM 숫자 보존 불일치 원본 대조
- [ ] wide/long 행 수와 컬럼 스키마 확인
- [ ] 최종 CSV `utf-8-sig` 저장 확인
- [ ] 커널 재시작 후 전체 실행
- [ ] 리팩터링 전후 QA 수치 비교